# Two-stage HB diagnostic notebook

This notebook lets you inspect one portfolio year without running `run_main.py`.

It checks:
- the estimation window used for a chosen year
- the external CFO model inputs
- `CFO_lead1_pred_scaled` after external prediction
- whether NaNs / infs remain before the accrual model
- whether the accrual model can be built and whether its initial point is valid


In [39]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pymc as pm


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'data').exists() and (p / 'results').exists():
            return p
    raise FileNotFoundError("Could not find project root containing 'data' and 'results'.")


PROJECT_ROOT = find_project_root()
SHARED_DIR = PROJECT_ROOT / 'modelling' / 'shared'
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SHARED_DIR   =', SHARED_DIR)


PROJECT_ROOT = /Users/sondregrontvedt/Documents/Industriell Økonomi og Teknologiledelse/excel_extract_project
SHARED_DIR   = /Users/sondregrontvedt/Documents/Industriell Økonomi og Teknologiledelse/excel_extract_project/modelling/shared


In [40]:
from hb_shared_utils import (
    compute_wca,
    build_regressors,
    assign_indices,
    mark_usable,
    build_estimation_window,
)

from uncertainty_model_hb import (
    fit_external_cfo_ar1_model,
    predict_cfo_lead_for_portfolio_rows,
    build_hb_accrual_model_fixed_lead,
)


In [41]:
# --- Settings ---
INPUT_CSV = PROJECT_ROOT / 'results' / 'extraction_static' / 'prepared_step2_input.csv'

PORT_YEAR = 2022
MIN_TRAIN_YEARS = 3
MAX_TRAIN_YEARS = 5

# Smaller CFO model for diagnostics
RANDOM_SEED = 42
CFO_DRAWS = 300
CFO_TUNE = 500
N_CHAINS = 4
TARGET_ACCEPT = 0.95
CFO_PREDICTION_MODE = 'mean'   # 'mean' or 'draw'

print('INPUT_CSV =', INPUT_CSV)
print('PORT_YEAR =', PORT_YEAR)


INPUT_CSV = /Users/sondregrontvedt/Documents/Industriell Økonomi og Teknologiledelse/excel_extract_project/results/extraction_static/prepared_step2_input.csv
PORT_YEAR = 2022


In [42]:
# --- Load and prepare panel exactly like Step 2 ---
data = pd.read_csv(INPUT_CSV)
data = compute_wca(data)
data = build_regressors(data, include_lead=True)
data = mark_usable(data)
data, firm_map, sector_map, firm_sector = assign_indices(data)

print('Panel shape:', data.shape)
print('Years:', int(data['Year'].min()), 'to', int(data['Year'].max()))
print('Usable rows:', int(data['usable'].sum()))
print('Unique firms:', int(data['Ticker'].nunique()))
print('CFO_lead1_scaled non-null:', int(data['CFO_lead1_scaled'].notna().sum()))


Panel shape: (10375, 43)
Years: 1988 to 2024
Usable rows: 9739
Unique firms: 634
CFO_lead1_scaled non-null: 9107


In [43]:
# --- Build the exact estimation window for the chosen portfolio year ---
window_df = build_estimation_window(
    data,
    PORT_YEAR,
    min_train_years=MIN_TRAIN_YEARS,
    max_train_years=MAX_TRAIN_YEARS,
    include_portfolio_year=True,
)

if window_df is None:
    raise ValueError(f'No estimation window could be built for PORT_YEAR={PORT_YEAR}')

print('window_df shape:', window_df.shape)
print('window years:', sorted(window_df['Year'].dropna().astype(int).unique().tolist()))
print('unique firms in window:', int(window_df['Ticker'].nunique()))
print('portfolio-year rows:', int(window_df['is_portfolio_year'].sum()))
print('training rows:', int((~window_df['is_portfolio_year']).sum()))

display(window_df[['Ticker','Year','is_portfolio_year','CFO_scaled','CFO_lead1_scaled','WCA_scaled']].head(20))


  Firms removed (<3 training obs): 46 / 617
window_df shape: (3359, 44)
window years: [2017, 2018, 2019, 2020, 2021, 2022]
unique firms in window: 571
portfolio-year rows: 570
training rows: 2789


,Ticker,Year,is_portfolio_year,CFO_scaled,CFO_lead1_scaled,WCA_scaled
0,20202.OL,2018,False,-0.011377,0.125115,-0.005828
1,20202.OL,2019,False,0.029143,0.133253,-0.008944
2,20202.OL,2020,False,0.069789,0.244125,-0.002793
3,20202.OL,2021,False,0.210021,0.118929,0.001355
4,8TRA.ST,2017,False,0.017026,0.009219,0.044641
5,8TRA.ST,2018,False,0.008330,0.024328,0.132670
6,8TRA.ST,2019,False,0.023693,0.047130,0.076971
7,8TRA.ST,2020,False,0.047738,0.034925,-0.021133
8,8TRA.ST,2021,False,0.031180,-0.013325,-0.068872
9,AAB.CO,2017,False,0.123607,-0.059990,0.468147


In [44]:
# --- Missingness before external CFO prediction ---
pre_cols = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

print('Missing values in raw window:')
display(window_df[pre_cols].isna().sum().to_frame('n_missing'))

bad_pre = window_df[pre_cols].isna().any(axis=1)
print('Rows with any missing among pre_cols:', int(bad_pre.sum()))
display(window_df.loc[bad_pre, ['Ticker','Year','is_portfolio_year'] + pre_cols].head(20))


Missing values in raw window:


,n_missing
WCA_scaled,0
CFO_lag1_scaled,0
CFO_scaled,0
CFO_lead1_scaled,1
dREV_scaled,0
PPE_scaled,0


Rows with any missing among pre_cols: 1


,Ticker,Year,is_portfolio_year,WCA_scaled,CFO_lag1_scaled,CFO_scaled,CFO_lead1_scaled,dREV_scaled,PPE_scaled
1000,FRO.OL,2021,False,0.007383,0.162726,0.015483,NaN,-0.144607,0.876736


In [45]:
# --- Fit external CFO model only ---
cfo_info, cfo_trace = fit_external_cfo_ar1_model(
    window_df=window_df,
    random_seed=RANDOM_SEED + 10_000 + int(PORT_YEAR),
    draws=CFO_DRAWS,
    tune=CFO_TUNE,
    chains=N_CHAINS,
    target_accept=TARGET_ACCEPT,
)

print('External CFO model fitted.')
print('n_ar1_obs =', cfo_info.get('n_ar1_obs'))
print('n_latent  =', cfo_info.get('n_latent', 'not found'))
print('keys      =', sorted(cfo_info.keys()))


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_cfo_market, rho_cfo_market, psi_cfo_market, sigma_mu_cfo, sigma_rho_cfo, mu_cfo_sector_raw, rho_cfo_sector_raw, psi_cfo_sector_raw, nu_cfo]


Output()

Sampling 4 chains for 500 tune and 300 draw iterations (2_000 + 1_200 draws total) took 26 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


External CFO model fitted.
n_ar1_obs = 2788
n_latent  = 571
keys      = ['ar1_obs_mask', 'cfo_curr', 'firm_to_sector', 'latent_idx', 'n_ar1_obs', 'n_latent', 'sector_of_obs', 'sector_remap', 'window_firms', 'window_sectors']


In [46]:
# --- Predict CFO_{t+1} externally ---
window_df_fixed = predict_cfo_lead_for_portfolio_rows(
    window_df=window_df,
    cfo_info=cfo_info,
    cfo_trace=cfo_trace,
    prediction_mode=CFO_PREDICTION_MODE,
)

check_cols = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_scaled',
    'CFO_lead1_pred_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

print('Missing values after external CFO prediction:')
display(window_df_fixed[check_cols].isna().sum().to_frame('n_missing'))

finite_flags = np.isfinite(window_df_fixed[check_cols].to_numpy(dtype=float, copy=True))
print('All finite in check_cols:', bool(finite_flags.all()))

bad_post = window_df_fixed[check_cols].isna().any(axis=1) | (~np.isfinite(window_df_fixed[check_cols].to_numpy(dtype=float, copy=True)).all(axis=1))
print('Rows with any missing/non-finite after prediction:', int(bad_post.sum()))
display(window_df_fixed.loc[bad_post, ['Ticker','Year','is_portfolio_year'] + check_cols].head(50))


Missing values after external CFO prediction:


,n_missing
WCA_scaled,0
CFO_lag1_scaled,0
CFO_scaled,0
CFO_lead1_scaled,1
CFO_lead1_pred_scaled,0
dREV_scaled,0
PPE_scaled,0


All finite in check_cols: False
Rows with any missing/non-finite after prediction: 1


,Ticker,Year,is_portfolio_year,WCA_scaled,CFO_lag1_scaled,CFO_scaled,CFO_lead1_scaled,CFO_lead1_pred_scaled,dREV_scaled,PPE_scaled
1000,FRO.OL,2021,False,0.007383,0.162726,0.015483,NaN,0.037443,-0.144607,0.876736


In [47]:
# --- Compare original vs predicted CFO lead on portfolio rows ---
portfolio_rows = window_df_fixed['is_portfolio_year'].fillna(False)
compare_cols = ['Ticker','Year','CFO_scaled','CFO_lead1_scaled','CFO_lead1_pred_scaled']
display(window_df_fixed.loc[portfolio_rows, compare_cols].head(30))


,Ticker,Year,CFO_scaled,CFO_lead1_scaled,CFO_lead1_pred_scaled
2789,20202.OL,2022,0.113507,0.121420,0.113343
2790,8TRA.ST,2022,-0.011460,0.050688,0.033022
2791,AAB.CO,2022,-0.312518,-0.224132,-0.187836
2792,AAK.ST,2022,-0.002373,0.180938,0.010084
2793,ABB.ST,2022,0.033506,0.122658,0.061923
2794,ABL.OL,2022,0.164182,0.108019,0.165325
2795,ACADE.ST,2022,0.154902,0.157957,0.139949
2796,ACELP.ST,2022,-0.448582,-0.475605,-0.473008
2797,ACG1V.HE,2022,0.113242,0.181398,0.134341
2798,ACRIa.ST,2022,-0.007975,0.037740,0.011925


In [48]:
# --- Optional safeguard used only for diagnostics ---
# Drop rows with missing regressors and inspect how many would be removed.
needed = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_pred_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

diag_bad = window_df_fixed[needed].isna().any(axis=1)
print('Rows that would be dropped before accrual model:', int(diag_bad.sum()))

window_df_model = window_df_fixed.loc[~diag_bad].copy()
print('window_df_model shape:', window_df_model.shape)
display(window_df_model[['Ticker','Year','is_portfolio_year'] + needed].head(20))


Rows that would be dropped before accrual model: 0
window_df_model shape: (3359, 45)


,Ticker,Year,is_portfolio_year,WCA_scaled,CFO_lag1_scaled,CFO_scaled,CFO_lead1_pred_scaled,dREV_scaled,PPE_scaled
0,20202.OL,2018,False,-0.005828,0.000000,-0.011377,0.125115,0.000000,1.647241
1,20202.OL,2019,False,-0.008944,-0.002650,0.029143,0.133253,0.052000,1.076466
2,20202.OL,2020,False,-0.002793,0.015263,0.069789,0.244125,0.129308,1.086343
3,20202.OL,2021,False,0.001355,0.060039,0.210021,0.118929,0.142753,0.929358
4,8TRA.ST,2017,False,0.044641,0.017697,0.017026,0.009219,0.059799,0.148554
5,8TRA.ST,2018,False,0.132670,0.015385,0.008330,0.024328,0.049051,0.123066
6,8TRA.ST,2019,False,0.076971,0.008113,0.023693,0.047130,0.035172,0.147775
7,8TRA.ST,2020,False,-0.021133,0.023998,0.047738,0.034925,-0.050876,0.162143
8,8TRA.ST,2021,False,-0.068872,0.042619,0.031180,-0.013325,0.138058,0.160063
9,AAB.CO,2017,False,0.468147,0.068016,0.123607,-0.059990,-0.083177,0.251613


In [49]:
# --- Build accrual model only ---
model, trace_info = build_hb_accrual_model_fixed_lead(window_df_model, firm_sector)
print('Accrual model built.')
print(trace_info)


Accrual model built.
{'window_firms': [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(31), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(75), np.i

In [50]:
# --- Check initial point before sampling ---
with model:
    start = model.initial_point()
    print('Initial point keys:', list(start.keys())[:10], '...')

    try:
        check = model.check_start_vals(start)
        print('Initial point check passed.')
        print(check)
    except Exception as e:
        print('Initial point check failed:')
        print(type(e).__name__, e)
        print('\nRunning model.debug() ...')
        model.debug()


Initial point keys: ['mu_0', 'omega_log__', 'tau_log__', 'sigma_0_log__', 'alpha_sector_raw', 'alpha_firm_raw', 'sigma_sector_raw_log__', 'sigma_firm_raw_log__', 'beta_CFO_lag1', 'beta_CFO_curr'] ...
Initial point check passed.
None


In [51]:
# --- Tiny smoke-test sample if the initial point is OK ---
# Keep this very small for diagnostics.
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    with model:
        trace = pm.sample(
            draws=50,
            tune=50,
            chains=2,
            target_accept=0.9,
            random_seed=RANDOM_SEED + int(PORT_YEAR),
            return_inferencedata=True,
            progressbar=True,
        )
    print(trace)
else:
    print('RUN_SMOKE_TEST = False, so no accrual sampling was run.')


RUN_SMOKE_TEST = False, so no accrual sampling was run.
